## **Training a Linear Regression Model Using PyTorch** 


The objective is to move from manual NumPy computations to building a full, automated training pipeline using PyTorch. We train a regression model on the scikit-learn Diabetes dataset under the following specified hyperparameters:
___

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 4</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Train a linear regression model in PyTorch using a regression dataset. Use the following parameters.
</p>

<ul>
    <li>Criterion: MSE Loss</li>
    <li>Fully Connected Layers x 2</li>
    <li>Batch Size: 8</li>
    <li>Optimizer: SGD</li>
    <li>Epoch: 1000</li>
</ul>
</div>

In [1]:
import sys
print(sys.executable)

C:\Users\keavy\anaconda3\envs\dlenv\python.exe


In [11]:
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

We set a random seed so that the results are reproducible, and we define the device that PyTorch will use for computation. The code will use the GPU if one is available; otherwise, it will use the CPU.

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed_everything()

We then load the Diabetes dataset and organize it into a DataFrame. The dataset contains 442 observations, with 10 feature variables and one target variable.

In [12]:
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y
df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   feature_0  442 non-null    float64
 1   feature_1  442 non-null    float64
 2   feature_2  442 non-null    float64
 3   feature_3  442 non-null    float64
 4   feature_4  442 non-null    float64
 5   feature_5  442 non-null    float64
 6   feature_6  442 non-null    float64
 7   feature_7  442 non-null    float64
 8   feature_8  442 non-null    float64
 9   feature_9  442 non-null    float64
 10  target     442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


The data is then divided into training and testing sets, with 80 percent used for training and 20 percent reserved for testing. We also standardize the features and target so that the values are on a comparable scale, which helps make the SGD training process more stable.

In [4]:
# Data 
X, y = load_diabetes(return_X_y=True)      
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# scale features and target so SGD trains stably
x_scaler, y_scaler = StandardScaler(), StandardScaler()
X_train = x_scaler.fit_transform(X_train)
X_test  = x_scaler.transform(X_test)
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test  = y_scaler.transform(y_test.reshape(-1, 1))

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test,  dtype=torch.float32).to(device)

The training data is converted into a TensorDataset and loaded in batches of 8. The data is shuffled so that the model does not always see the training examples in the same order.

In [5]:
# Dataset and DataLoader (batch size 8) 
train_ds = TensorDataset(X_train, y_train)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)

For the model, we use two fully connected layers. The first layer takes the 10 input features and connects them to 16 neurons, followed by ReLU activation. The second layer reduces those 16 values to a single output, which represents the predicted regression value.

In [6]:
# Model: 2 fully connected layers 
model = nn.Sequential(
    nn.Linear(10, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
).to(device)

For training, we use Mean Squared Error as the loss function because this is a regression problem. We use Stochastic Gradient Descent as the optimizer with a learning rate of 0.01.

In [7]:
# Loss and Optimizer 
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

The model is now trained for 1000 epochs. During each epoch, the model makes predictions through the forward pass, calculates the MSE loss, performs backpropagation to calculate the gradients, and then uses SGD to update the model parameters. The training MSE is displayed every 100 epochs so we can observe how the loss changes throughout training.

In [8]:
# Train (1000 epochs)
epochs = 1000
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)

        preds = model(xb)               # forward pass
        loss = criterion(preds, yb)     # MSE

        optimizer.zero_grad()           # clear old gradients
        loss.backward()                 # backpropagation
        optimizer.step()                # SGD weight update

        running_loss += loss.item() * xb.size(0)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}]  Train MSE: {running_loss / len(train_ds):.4f}")

Epoch [100/1000]  Train MSE: 0.4000
Epoch [200/1000]  Train MSE: 0.3400
Epoch [300/1000]  Train MSE: 0.2959
Epoch [400/1000]  Train MSE: 0.2861
Epoch [500/1000]  Train MSE: 0.2707
Epoch [600/1000]  Train MSE: 0.2560
Epoch [700/1000]  Train MSE: 0.2408
Epoch [800/1000]  Train MSE: 0.2278
Epoch [900/1000]  Train MSE: 0.2263
Epoch [1000/1000]  Train MSE: 0.2728


>The training MSE generally decreases from 0.4000 at epoch 100 to 0.2263 at epoch 900, before increasing to 0.2728 at epoch 1000. This shows that the model was learning from the training data, although the final training loss was slightly higher than the value observed at epoch 900.

In [9]:
# ---------- Evaluate ----------
model.eval()
with torch.no_grad():
    test_loss = criterion(model(X_test), y_test)
print(f"Test MSE: {test_loss.item():.4f}")

Test MSE: 0.5948


>After training, we evaluate the model using the test set, which was not used during training. The final test MSE is 0.5948, which gives us a measure of the model's prediction error on unseen data.


Overall, this activity demonstrates how a regression model can be trained automatically using PyTorch. We prepared and standardized the Diabetes dataset, built a two-layer fully connected model, trained it using MSE loss and SGD for 1000 epochs, and evaluated it on the test data. The final test MSE was 0.5948, providing the model's error on data that was not used during training